# Module 10 Lab — Multi-Agent Governance & Delegation

**Scenario:** A procurement team of agents delegates work from a manager to research and procurement specialists, with a payment executor and verifier.

We will make **authority an explicit data structure** instead of assuming that a message from one agent authorizes another.

In [ ]:
%pip install -q "pydantic>=2" pandas
print("Dependencies installed.")

In [ ]:
from __future__ import annotations
from pydantic import BaseModel, Field
from datetime import datetime, timezone, timedelta
from typing import Optional, Any
from uuid import uuid4
import pandas as pd, hashlib, json
pd.set_option("display.max_colwidth",120)

## 1. Workload identities

In [ ]:
class AgentIdentity(BaseModel):
    agent_id:str
    role:str
    tools:set[str]
    resources:set[str]
    max_spend:float=0
    may_delegate:bool=False
    max_delegation_depth:int=0

AGENTS={
 "manager":AgentIdentity(agent_id="agent:manager",role="manager",
   tools={"vendor.search","po.request"},resources={"vendor-catalog","procurement"},
   max_spend=25000,may_delegate=True,max_delegation_depth=2),
 "research":AgentIdentity(agent_id="agent:research",role="research",
   tools={"vendor.search","vendor.read"},resources={"vendor-catalog"},
   max_spend=0,may_delegate=False,max_delegation_depth=0),
 "procurement":AgentIdentity(agent_id="agent:procurement",role="procurement",
   tools={"vendor.read","po.create"},resources={"vendor-catalog","procurement"},
   max_spend=25000,may_delegate=True,max_delegation_depth=1),
 "payment":AgentIdentity(agent_id="agent:payment",role="payment",
   tools={"payment.execute"},resources={"payments"},
   max_spend=25000,may_delegate=False,max_delegation_depth=0),
}
pd.DataFrame([a.model_dump() for a in AGENTS.values()])

## 2. Delegation contract

In [ ]:
class DelegationGrant(BaseModel):
    delegation_id:str=Field(default_factory=lambda:f"del-{uuid4().hex[:8]}")
    parent_id:Optional[str]=None
    issuer:str
    subject:str
    on_behalf_of:str
    task_id:str
    purpose:str
    allowed_tools:set[str]
    allowed_resources:set[str]
    max_spend:float=0
    remaining_calls:int=20
    remaining_depth:int=0
    issued_at:datetime=Field(default_factory=lambda:datetime.now(timezone.utc))
    expires_at:datetime=Field(default_factory=lambda:datetime.now(timezone.utc)+timedelta(minutes=30))
    policy_version:str="v1"
    trace_id:str=Field(default_factory=lambda:f"trace-{uuid4().hex[:8]}")
    revoked:bool=False

## 3. Root authority from the human/task

In [ ]:
root=DelegationGrant(
 issuer="user:123",subject="agent:manager",on_behalf_of="user:123",
 task_id="task:buy-laptops",purpose="approved_procurement",
 allowed_tools={"vendor.search","vendor.read","po.request","po.create","payment.execute"},
 allowed_resources={"vendor-catalog","procurement","payments"},
 max_spend=20000,remaining_calls=50,remaining_depth=2
)
root

## 4. Effective child authority = intersection

In [ ]:
def issue_child(parent:DelegationGrant, child_key:str, *,
                tools:set[str],resources:set[str],max_spend:float,
                purpose:Optional[str]=None,ttl_minutes=15)->DelegationGrant:
    child=AGENTS[child_key]
    if parent.revoked: raise PermissionError("Parent grant revoked")
    if datetime.now(timezone.utc)>parent.expires_at: raise PermissionError("Parent grant expired")
    if parent.remaining_depth<=0: raise PermissionError("Delegation depth exhausted")
    if not tools <= parent.allowed_tools: raise PermissionError("Privilege amplification: tool")
    if not tools <= child.tools: raise PermissionError("Child role does not own requested tool")
    if not resources <= parent.allowed_resources: raise PermissionError("Privilege amplification: resource")
    if not resources <= child.resources: raise PermissionError("Child role does not own resource")
    if max_spend>parent.max_spend or max_spend>child.max_spend:
        raise PermissionError("Privilege amplification: spend")
    return DelegationGrant(
      parent_id=parent.delegation_id,issuer=parent.subject,subject=child.agent_id,
      on_behalf_of=parent.on_behalf_of,task_id=parent.task_id,
      purpose=purpose or parent.purpose,allowed_tools=tools,allowed_resources=resources,
      max_spend=max_spend,remaining_calls=min(parent.remaining_calls,20),
      remaining_depth=min(parent.remaining_depth-1,child.max_delegation_depth),
      expires_at=min(parent.expires_at,datetime.now(timezone.utc)+timedelta(minutes=ttl_minutes)),
      policy_version=parent.policy_version,trace_id=parent.trace_id)

research_grant=issue_child(root,"research",tools={"vendor.search","vendor.read"},
                           resources={"vendor-catalog"},max_spend=0)
research_grant

## 5. Attempt privilege amplification

In [ ]:
tests=[
 ("payment tool to research",dict(child_key="research",tools={"payment.execute"},resources={"vendor-catalog"},max_spend=0)),
 ("overspend procurement",dict(child_key="procurement",tools={"po.create"},resources={"procurement"},max_spend=50000)),
]
for name,kwargs in tests:
    try:
        issue_child(root,**kwargs)
        print(name,"UNEXPECTED ALLOW")
    except PermissionError as e:
        print(name,"DENIED:",e)

## 6. Delegation graph

In [ ]:
GRANTS={root.delegation_id:root,research_grant.delegation_id:research_grant}

def add_grant(g):
    GRANTS[g.delegation_id]=g
    return g

proc_grant=add_grant(issue_child(root,"procurement",tools={"vendor.read","po.create"},
                                 resources={"vendor-catalog","procurement"},max_spend=15000))
pd.DataFrame([{
 "id":g.delegation_id,"parent":g.parent_id,"issuer":g.issuer,"subject":g.subject,
 "purpose":g.purpose,"max_spend":g.max_spend,"depth":g.remaining_depth
} for g in GRANTS.values()])

## 7. Structured handoff envelope

In [ ]:
class Handoff(BaseModel):
    handoff_id:str=Field(default_factory=lambda:f"ho-{uuid4().hex[:8]}")
    from_agent:str
    to_agent:str
    delegation_id:str
    task_id:str
    purpose:str
    payload:dict[str,Any]
    created_at:datetime=Field(default_factory=lambda:datetime.now(timezone.utc))

handoff=Handoff(
 from_agent="agent:manager",to_agent="agent:research",
 delegation_id=research_grant.delegation_id,task_id=root.task_id,
 purpose="vendor_due_diligence",
 payload={"vendor_id":"V-42","requested_output":"risk_summary"}
)
handoff

## 8. Validate handoff against grant

In [ ]:
def validate_handoff(h:Handoff)->tuple[bool,str]:
    g=GRANTS.get(h.delegation_id)
    if not g:return False,"Unknown delegation"
    if g.revoked:return False,"Grant revoked"
    if g.subject!=h.to_agent:return False,"Wrong recipient"
    if g.issuer!=h.from_agent:return False,"Wrong issuer"
    if g.task_id!=h.task_id:return False,"Task mismatch"
    if datetime.now(timezone.utc)>g.expires_at:return False,"Grant expired"
    return True,"Valid handoff"
validate_handoff(handoff)

## 9. Context minimization

In [ ]:
upstream_context={
 "vendor_id":"V-42",
 "vendor_name":"Northstar",
 "research_question":"Assess delivery and vendor risk",
 "employee_salary_data":"CONFIDENTIAL HR DATA",
 "payment_api_secret":"sk-secret",
 "customer_records":["..."],
}
ALLOWED_CONTEXT_BY_ROLE={
 "research":{"vendor_id","vendor_name","research_question"},
 "procurement":{"vendor_id","vendor_name"},
}
def minimize_context(context,role):
    allowed=ALLOWED_CONTEXT_BY_ROLE[role]
    return {k:v for k,v in context.items() if k in allowed}
minimize_context(upstream_context,"research")

## 10. Confused-deputy prevention

In [ ]:
def authorize_action(grant_id:str,actor:str,tool:str,resource:str,amount=0)->tuple[bool,str]:
    g=GRANTS.get(grant_id)
    if not g:return False,"No delegation"
    if g.revoked:return False,"Revoked"
    if g.subject!=actor:return False,"Actor does not own grant"
    if datetime.now(timezone.utc)>g.expires_at:return False,"Expired"
    if tool not in g.allowed_tools:return False,"Tool outside delegated authority"
    if resource not in g.allowed_resources:return False,"Resource outside delegated authority"
    if amount>g.max_spend:return False,"Amount outside delegated authority"
    return True,"Authorized by delegation"

# Research agent cannot cause a PO merely by asking procurement in natural language.
print(authorize_action(research_grant.delegation_id,"agent:research","po.create","procurement",1000))
print(authorize_action(proc_grant.delegation_id,"agent:procurement","po.create","procurement",1000))

## 11. Shared aggregate budget

In [ ]:
class SharedBudget:
    def __init__(self,limit): self.limit=limit; self.used=0
    def reserve(self,amount):
        if self.used+amount>self.limit:
            return False,f"Global budget exceeded: {self.used}+{amount}>{self.limit}"
        self.used+=amount
        return True,f"Reserved; total={self.used}"

budget=SharedBudget(20000)
for worker,amt in [("A",8000),("B",8000),("C",8000)]:
    print(worker,budget.reserve(amt))

## 12. Idempotency across workers

In [ ]:
EXECUTED={}
def execute_once(action_key:str,actor:str,result_factory):
    if action_key in EXECUTED:
        return {"duplicate":True,"result":EXECUTED[action_key]}
    result=result_factory()
    EXECUTED[action_key]=result
    return {"duplicate":False,"result":result}

def fake_po(): return {"po_id":f"PO-{uuid4().hex[:6]}","status":"created"}
print(execute_once("task:buy-laptops:po:V-42","agent:procurement",fake_po))
print(execute_once("task:buy-laptops:po:V-42","agent:procurement",fake_po))

## 13. Descendant revocation

In [ ]:
def descendants(grant_id):
    found=[]
    frontier=[grant_id]
    while frontier:
        p=frontier.pop()
        kids=[g.delegation_id for g in GRANTS.values() if g.parent_id==p]
        found.extend(kids); frontier.extend(kids)
    return found

def revoke_tree(grant_id):
    ids=[grant_id]+descendants(grant_id)
    for i in ids:
        if i in GRANTS: GRANTS[i].revoked=True
    return ids

print("descendants:",descendants(root.delegation_id))

## 14. Global halt state

In [ ]:
class ControlPlane:
    def __init__(self): self.state="RUNNING"
    def set_state(self,state):
        assert state in {"RUNNING","PAUSED","TERMINATING","TERMINATED"}
        self.state=state
    def may_act(self):
        return self.state=="RUNNING"

control=ControlPlane()
print(control.may_act())
control.set_state("PAUSED")
print(control.may_act())

## 15. Every consequential action checks both authority and global state

In [ ]:
def governed_execute(grant_id,actor,tool,resource,amount,action_key):
    if not control.may_act():
        return {"allowed":False,"reason":f"Global state={control.state}"}
    ok,reason=authorize_action(grant_id,actor,tool,resource,amount)
    if not ok:return {"allowed":False,"reason":reason}
    return {"allowed":True,"execution":execute_once(action_key,actor,lambda:{"status":"executed","amount":amount})}

control.set_state("RUNNING")
governed_execute(proc_grant.delegation_id,"agent:procurement","po.create","procurement",5000,"po:V-42")

## 16. Multi-agent evidence trail

In [ ]:
EVENTS=[]
def record(event_type,actor,**kwargs):
    EVENTS.append({
      "ts":datetime.now(timezone.utc).isoformat(),
      "trace_id":root.trace_id,"task_id":root.task_id,
      "type":event_type,"actor":actor,**kwargs
    })

record("delegation","agent:manager",delegation_id=research_grant.delegation_id,to="agent:research")
record("handoff","agent:manager",handoff_id=handoff.handoff_id,to="agent:research")
record("agent_result","agent:research",result="vendor risk low")
record("delegation","agent:manager",delegation_id=proc_grant.delegation_id,to="agent:procurement")
record("action","agent:procurement",tool="po.create",amount=5000)
display(pd.DataFrame(EVENTS))

## 17. Delegation chain reconstruction

In [ ]:
def chain(grant_id):
    out=[]
    current=GRANTS.get(grant_id)
    while current:
        out.append(current)
        current=GRANTS.get(current.parent_id) if current.parent_id else None
    return list(reversed(out))
[(g.issuer,g.subject,g.delegation_id) for g in chain(proc_grant.delegation_id)]

## 18. Governance metrics

In [ ]:
metrics={
 "max_delegation_depth_used":max(len(chain(g.delegation_id))-1 for g in GRANTS.values()),
 "active_grants":sum(not g.revoked and g.expires_at>datetime.now(timezone.utc) for g in GRANTS.values()),
 "revoked_grants":sum(g.revoked for g in GRANTS.values()),
 "trace_events":len(EVENTS),
 "duplicate_actions":0, # wire from execution telemetry in production
}
metrics

## 19. Adversarial regression suite

In [ ]:
checks=[]
try:
    issue_child(root,"research",tools={"payment.execute"},resources={"payments"},max_spend=1000)
    checks.append(("privilege amplification blocked",False))
except PermissionError:
    checks.append(("privilege amplification blocked",True))

checks.append(("cross-role action blocked",
 authorize_action(research_grant.delegation_id,"agent:research","po.create","procurement",100)[0] is False))

bad=handoff.model_copy(update={"to_agent":"agent:payment"})
checks.append(("handoff recipient binding",validate_handoff(bad)[0] is False))

control.set_state("PAUSED")
checks.append(("global halt blocks action",
 governed_execute(proc_grant.delegation_id,"agent:procurement","po.create","procurement",100,"x")["allowed"] is False))
control.set_state("RUNNING")

expired=proc_grant.model_copy(update={"expires_at":datetime.now(timezone.utc)-timedelta(seconds=1)})
GRANTS["expired-test"]=expired
checks.append(("expired grant blocked",
 authorize_action("expired-test","agent:procurement","po.create","procurement",100)[0] is False))

df=pd.DataFrame(checks,columns=["test","pass"]); display(df); assert df["pass"].all()

## 20. OpenAI Agents SDK — manager pattern

Current SDK pattern:

```python
from agents import Agent

research = Agent(
    name="Research",
    instructions="Perform bounded vendor research."
)

manager = Agent(
    name="Procurement Manager",
    instructions="Coordinate procurement. Never grant authority through prose.",
    tools=[
        research.as_tool(
            tool_name="research_vendor",
            tool_description="Research a vendor; no execution authority."
        )
    ]
)
```

The framework provides composition. Your governance layer should still issue and validate the delegation contract before a specialist receives consequential tools or data.

## 21. OpenAI Agents SDK — handoff pattern

```python
from agents import Agent, handoff

specialist = Agent(name="Procurement Specialist", instructions="...")
triage = Agent(
    name="Triage",
    handoffs=[handoff(specialist)]
)
```

Use handoff input schemas/filters to reduce transferred context. Keep authorization separate from conversational control transfer.

## 22. Microsoft Agent Framework mapping

Current Agent Framework includes:

```text
Sequential
Concurrent
Handoff
Group Chat
Magentic
```

The same governance model can be applied to each orchestration:

- every participant has workload identity,
- every edge has allowed delegation semantics,
- global budgets live outside individual agents,
- checkpointed state contains authority references,
- tool approval binds to the final action,
- halt state is shared.

For handoff specifically, distinguish **task ownership transfer** from **authority transfer**.

# 23. Exercises

### A — Payment delegation
Allow Procurement to delegate a payment up to CAD 10K to Payment, but never above its own remaining grant.

### B — Recursive delegation
Create a third-level worker and prove depth exhaustion works.

### C — Purpose change
Attempt to reuse a procurement grant for HR research.

### D — Context leak
Build a handoff filter that removes secrets and unrelated sensitive fields.

### E — Parallel workers
Run simulated concurrent reservations against one global budget.

### F — Revocation race
Revoke a parent while a child is preparing an action.

### G — Global kill
Verify every worker checks shared halt state before side effects.

### H — Disagreement
Create planner and risk-agent disagreement and route high-impact conflicts to human review.

### I — OpenAI Agents SDK
Implement manager and handoff variants of the same scenario and compare governance surfaces.

### J — Microsoft Agent Framework
Implement sequential vs handoff orchestration and document how task ownership changes.

### K — Evidence
Produce a full audit record from human request to final tool action.

# 24. Key takeaways

1. Multi-agent governance is fundamentally about preserving authority across distributed decisions.
2. Delegation is not the same as credential sharing or impersonation.
3. Child authority must be an intersection of parent grant, child role, task, and runtime policy.
4. Bind delegation to purpose, task, resources, tools, budgets, TTL, and depth.
5. Keep actor identity distinct from the human represented.
6. Minimize context at every handoff.
7. Treat inter-agent messages as inputs, not proof of authority.
8. Prevent confused-deputy behavior at downstream agents.
9. Shared risks need shared controls: budget, state, halt, audit.
10. Make consequential operations idempotent.
11. Propagate revocation through descendants.
12. Bind human approval to the final action and delegation lineage.
13. Store a delegation graph for audit and revocation.
14. Consensus between agents is not an authorization mechanism.
15. Choose the simplest orchestration pattern that meets the enterprise need.